# Module 2 · Obtaining a Dataset

*Part of the Jupyter Book “Open Research in the Digital Humanities”. This is the **first** of three notebooks that share one dataset:*
**M2 obtain > M4 filter > M5 analyse.**

**What you do here:** get works by a creator from [Europeana](https://www.europeana.eu/), learn why the *same* artist hides under several catalogued **names**, combine and de-duplicate them, check the data is fit for purpose, and save it. (Analysis comes later, in Module 5.)

You can run this **without an API key** (a small shipped sample is used). To pull the full, live dataset, add your key in Step 1 below.


## Step 1: your Europeana API key

The next cell is the **only place** you paste your key.


In [1]:
# ==================================================================
#                  PASTE YOUR EUROPEANA API KEY HERE
# ==================================================================
#   1. Get a free key:  https://pro.europeana.eu/pages/get-api-keys
#   2. Replace the words  PASTE_YOUR_KEY_HERE  below (keep the quotes).
#
#        
API_KEY = "PASTE_YOUR_KEY_HERE"   # <-- your key goes here
#
# ==================================================================

_PLACEHOLDER = "PASTE_YOUR_KEY_HERE"
HAVE_KEY = API_KEY.strip() not in ("", _PLACEHOLDER)
print("API key detected." if HAVE_KEY else "No key set -> the shipped sample will be used.")

No key set -> the shipped sample will be used.


## Step 2: what to search for

Leave these as they are for your first run. `CREATOR_VARIANTS` is the heart of the lesson: the same artist is catalogued under different name forms, so we search several.


In [2]:
# The same creator, written the different ways catalogues store it:
CREATOR_VARIANTS = [
    "Vincent van Gogh",
    "Gogh, Vincent van",
    "Vincent Willem van Gogh",
    "van Gogh, Vincent",
]

# Use the live API only if a key is present; otherwise use the shipped sample.
DATA_SOURCE = "europeana" if HAVE_KEY else "course"
print("Data source:", DATA_SOURCE)
print("Searching variants:", CREATOR_VARIANTS)

Data source: course
Searching variants: ['Vincent van Gogh', 'Gogh, Vincent van', 'Vincent Willem van Gogh', 'van Gogh, Vincent']


In [3]:
from pathlib import Path
import pandas as pd

# Candidate locations for the shared datasets/ folder (repo root, or one level up from a module).
_DATASET_NAMES = ["van_gogh_combined.csv", "van_gogh_europeana.csv", "van_gogh_filtered.csv"]

def find_dataset(name):
    """Return the path to a dataset file, wherever the notebook is run from."""
    for base in [Path("datasets"), Path("../datasets")]:
        if (base / name).exists():
            return base / name
    raise FileNotFoundError(f"{name} not found in datasets/, run the earlier notebook first.")

def datasets_dir():
    """The datasets/ folder that actually holds the data (so saves land beside the real files)."""
    for base in [Path("datasets"), Path("../datasets")]:
        if any((base / n).exists() for n in _DATASET_NAMES):
            return base
    Path("datasets").mkdir(parents=True, exist_ok=True)   # first run, live mode
    return Path("datasets")

import requests
RAW = Path("data/raw"); RAW.mkdir(parents=True, exist_ok=True)

## 1 · The name-variation problem (authority control)

The *same* artist returns wildly different results depending on the exact string you search:

| Query | Records |
|---|---|
| `CREATOR:"Vincent van Gogh"` | 24 |
| `CREATOR:"Gogh, Vincent van"` | 309 |
| `who:"Vincent van Gogh"` | 354 |

So we search a **list of variants** and combine them, then **de-duplicate**, because one record can match several variants.

> **This problem gets solved for good in Module 6**, where we attach a stable **Wikidata Q-ID** to each entity. (See `TEACHING_THREADS.md`, Thread A.)

> **Prompt you could use:** *List plausible catalogue name variants for a creator, natural form, “surname, first name”, with/without middle names, common transliterations, so I can search each and combine.*


In [4]:
def _first(v):
    return (v[0] if v else None) if isinstance(v, list) else v

def fetch_variant(variant, api_key, max_pages=20, rows=100):
    """All records whose CREATOR exactly matches one name variant."""
    url = "https://api.europeana.eu/record/v2/search.json"
    out, cursor, pages = [], "*", 0
    while cursor and pages < max_pages:
        p = {"wskey": api_key, "query": "*:*", "qf": f'CREATOR:"{variant}"', "rows": rows, "cursor": cursor}
        r = requests.get(url, params=p, timeout=30); r.raise_for_status()
        d = r.json()
        for it in d.get("items", []):
            out.append({"id": it.get("id"), "title": _first(it.get("title")),
                        "type": _first(it.get("type")), "institution": _first(it.get("dataProvider")),
                        "country": _first(it.get("country")), "year": _first(it.get("year")),
                        "matched_variant": variant})
        cursor = d.get("nextCursor"); pages += 1
    return out

def fetch_all_variants(variants, api_key):
    rows = []
    for v in variants:
        got = fetch_variant(v, api_key); print(f"  {v!r}: {len(got)}"); rows += got
    df = pd.DataFrame(rows); before = len(df)
    df = df.drop_duplicates(subset="id").reset_index(drop=True)
    print(f"Combined {before} -> {len(df)} unique records")
    return df

def load_sample():
    # No-key default = the precomputed COMBINED result (what the live variant search produces),
    # so the Module 2 -> 4 -> 5 chain is continuous whether or not you have an API key.
    return pd.read_csv(find_dataset("van_gogh_combined.csv"))

## 2 · Get the data and save it

> **About the no-key run:** without a key the notebook loads the **combined dataset shipped with the course**, the *same result* the live variant search produces (≈334 records), so you can keep going and the next modules line up. The evidence table above is the point of the lesson: the obvious spelling alone finds only **24**.


In [5]:
if DATA_SOURCE == "europeana":
    print("Querying Europeana across name variants...")
    df = fetch_all_variants(CREATOR_VARIANTS, API_KEY)
    out = datasets_dir() / "van_gogh_combined.csv"
    df.to_csv(out, index=False); print("Saved combined dataset ->", out)
else:
    df = load_sample()
    print(f"Loaded the shipped combined dataset ({len(df)} records), the precomputed result of the "
          "variant search above. Add an API key to reproduce it live.")

df.to_csv(RAW / "dataset_raw.csv", index=False)
print(f"\nTotal records: {len(df)}"); df.head()

Loaded the shipped combined dataset (334 records), the precomputed result of the variant search above. Add an API key to reproduce it live.

Total records: 334


,id,title,type,institution,country,year,matched_variant
0,/2024903/photography_ProvidedCHO_KU_Leuven_999...,Vincent van Gogh. Self portrait as painter,IMAGE,Catholic University of Leuven,Belgium,1888.0,Vincent van Gogh
1,/2024903/photography_ProvidedCHO_KU_Leuven_999...,Vincent van Gogh. Korenveld met cypressen,IMAGE,Catholic University of Leuven,Belgium,1889.0,Vincent van Gogh
2,/2024903/photography_ProvidedCHO_KU_Leuven_999...,Vincent van Gogh. Olijfgaard,IMAGE,Catholic University of Leuven,Belgium,1889.0,Vincent van Gogh
3,/2024903/photography_ProvidedCHO_KU_Leuven_999...,Vincent van Gogh. Treurende oude man,IMAGE,Catholic University of Leuven,Belgium,1890.0,Vincent van Gogh
4,/2024903/photography_ProvidedCHO_KU_Leuven_999...,Vincent van Gogh. Landschap in de Provence bij...,IMAGE,Catholic University of Leuven,Belgium,1890.0,Vincent van Gogh


## 3 · Critical check, is one provider dominating?

Combining variants can drag in a single aggregator (for Van Gogh, a photographic *reproduction* index). Look before you trust, and **document** what you find. We don't drop anything here; that decision belongs to the filtering step (Module 4).


In [6]:
prov = df["institution"].value_counts()
print(prov.head(10))
if len(prov):
    share = prov.iloc[0] / len(df)
    print(f"\nTop provider {prov.index[0]!r} = {share:.0%} of records"
          + ("  <-- dominates; revisit in Module 4." if share > 0.5 else ""))

institution
German Documentation Center for Art History - Marburg Picture Index    300
Catholic University of Leuven                                           18
International Institute of Social History                                4
Digital Library for Dutch Literature                                     4
Austrian Gallery Belvedere                                               2
The Israel Museum, Jerusalem                                             1
Nationalmuseum Sweden                                                    1
Wellcome Collection                                                      1
Digital Memory of Catalonia                                              1
Finnish National Gallery                                                 1
Name: count, dtype: int64

Top provider 'German Documentation Center for Art History - Marburg Picture Index' = 90% of records  <-- dominates; revisit in Module 4.


## 4 · Is it fit for the question?


In [7]:
print("Records:", len(df))
print("\nMissing per column:\n", df.isna().mean().round(2))
print("\nUnique per column:\n", df.nunique())

Records: 334

Missing per column:
 id                 0.00
title              0.00
type               0.00
institution        0.00
country            0.00
year               0.92
matched_variant    0.00
dtype: float64

Unique per column:
 id                 334
title              316
type                 2
institution         11
country              9
year                 8
matched_variant      3
dtype: int64


## Done, dataset saved

You now have a raw dataset (`data/raw/dataset_raw.csv`, and `datasets/van_gogh_combined.csv` if you ran live).

**Workflow note (for your README):** *Searched Europeana across several name variants for Van Gogh, combined and de-duplicated the results; observed that one reproduction index dominates the combined set (to be handled in filtering).*

**Next: Module 4, Filtering the dataset.**
